# 09e Vehicle ReID: ResNet101-IBN-a VeRi-776 Pretraining

Pretrain the vehicle secondary backbone on VeRi-776 before CityFlowV2 fine-tuning.

Inputs:
- `mrkdagods/mtmc-weights`
- `abhyudaya12/veri-vehicle-re-identification-dataset`

Output:
- `/kaggle/working/resnet101ibn_veri776_best.pth`

In [ ]:
import os, sys, subprocess, shutil, json, re
from pathlib import Path

if shutil.which("nvidia-smi"):
    result = subprocess.run(
        ["nvidia-smi", "--query-gpu=gpu_name,compute_cap", "--format=csv,noheader"],
        capture_output=True, text=True, check=False,
    )
    if result.returncode == 0 and result.stdout.strip():
        gpu_name, capability = result.stdout.strip().split(",", 1)
        match = re.search(r"(\d+)\.(\d+)", capability)
        if match:
            sm = int(match.group(1)) * 10 + int(match.group(2))
            if sm < 70:
                subprocess.check_call([
                    sys.executable, "-m", "pip", "install", "-q",
                    "torch==2.4.1", "torchvision==0.19.1",
                    "--index-url", "https://download.pytorch.org/whl/cu124",
                ])

import torch
print(json.dumps({
    "python": sys.version.split()[0],
    "torch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device_count": torch.cuda.device_count(),
}, indent=2))

In [ ]:
REPO_URL = "https://github.com/MRKDaGods/gp.git"
WORK_DIR = Path("/kaggle/working")
PROJECT = WORK_DIR / "gp"

if not PROJECT.exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1", "--branch", "feature/people-tracking",
        REPO_URL, str(PROJECT),
    ])
else:
    subprocess.check_call(["git", "-C", str(PROJECT), "pull", "--ff-only"])

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
print(PROJECT)

In [ ]:
def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

requirements_path = PROJECT / "requirements.txt"
assert requirements_path.exists(), requirements_path
pip_install("-r", str(requirements_path))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"], cwd=str(PROJECT))

try:
    import torchreid  # noqa: F401
except ImportError:
    pip_install("git+https://github.com/KaiyangZhou/deep-person-reid.git")

for module_name in ["torchreid", "timm", "sklearn", "loguru", "omegaconf"]:
    __import__(module_name)
print("Dependencies installed")

In [ ]:
WEIGHTS_ROOT = Path("/kaggle/input/datasets/mrkdagods/mtmc-weights")
assert WEIGHTS_ROOT.exists(), "Attach the mtmc-weights dataset before running this notebook."

VERI_CANDIDATES = [
    Path("/kaggle/input/veri-vehicle-re-identification-dataset/VeRi"),
    Path("/kaggle/input/veri-vehicle-re-identification-dataset"),
    Path("/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset/VeRi"),
    Path("/kaggle/input/datasets/abhyudaya12/veri-vehicle-re-identification-dataset"),
    Path("/kaggle/input/datasets/mrkdagods/veri776/VeRi"),
    Path("/kaggle/input/datasets/mrkdagods/veri776"),
]
VERI_ROOT = None
for candidate in VERI_CANDIDATES:
    if (candidate / "image_train").exists() and (candidate / "image_query").exists() and (candidate / "image_test").exists():
        VERI_ROOT = candidate
        break
assert VERI_ROOT is not None, f"VeRi-776 not found. Tried: {VERI_CANDIDATES}"

OUTPUT_DIR = Path("/kaggle/working/09e_vehicle_reid_resnet101ibn_veri776")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_WEIGHTS = Path("/kaggle/working/resnet101ibn_veri776_best.pth")
print(json.dumps({
    "weights_root": str(WEIGHTS_ROOT),
    "veri_root": str(VERI_ROOT),
    "output_dir": str(OUTPUT_DIR),
}, indent=2))

In [ ]:
import shlex

train_cmd = [
    sys.executable, "-m", "src.training.train_reid",
    "--dataset", "veri776",
    "--root", str(VERI_ROOT),
    "--output-dir", str(OUTPUT_DIR),
    "--backbone", "resnet101_ibn_a",
    "--height", "384",
    "--width", "384",
    "--batch-size", "32",
    "--num-instances", "4",
    "--epochs", "120",
    "--lr", "3.5e-4",
    "--warmup-epochs", "10",
    "--scheduler", "cosine",
    "--loss", "triplet+center",
    "--triplet-margin", "0.3",
    "--center-weight", "0.0005",
    "--random-erasing", "0.5",
    "--color-jitter",
    "--eval-every", "10",
    "--fp16",
    "--num-workers", "2",
]
print(shlex.join(train_cmd))
subprocess.check_call(train_cmd, cwd=str(PROJECT))

best_weights = OUTPUT_DIR / "reid_veri776_resnet101_ibn_a_best.pth"
assert best_weights.exists(), f"Best VeRi-776 checkpoint not found: {best_weights}"
shutil.copy2(best_weights, FINAL_WEIGHTS)
print(f"Saved final checkpoint to {FINAL_WEIGHTS}")

In [ ]:
artifacts = sorted(str(path) for path in OUTPUT_DIR.glob("*"))
print(json.dumps({
    "final_weights": str(FINAL_WEIGHTS),
    "artifacts": artifacts,
}, indent=2))